In [ ]:
!pip install cobra mygene optlang highspy -q --upgrade
!apt-get install -y glpk-utils -q

# Load dependencies
import cobra
import pickle
import os
import numpy as np
from cobra.flux_analysis import pfba, flux_variability_analysis
from cobra.util.solver import linear_reaction_coefficients
from optlang.symbolics import Zero
import sys
import mygene
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ast
from cobra.core.gene import GPR
from ast import And, Or
import highspy
print("highspy OK") # confirms the HiGHS solver installed correctly - COBRApy's
                      # 'hybrid' solver mode (set below) can fall back to it
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import networkx as nx
import textwrap
import optlang
print(optlang.available_solvers) # sanity check: confirms which LP solvers
                                   # optlang can actually see in this environment


pd.set_option("display.max_colwidth", 55)
pd.set_option("display.width",        130) # widen pandas display so long
                                              # reaction/pathway names aren't truncated


cobra.Configuration().solver = "hybrid" # sets the default solver for all models
                                          # created from here on


In [ ]:
from google.colab import drive
drive.mount('/content/drive') # gives access to the Human-GEM model file and
                                # gem_input_data pickle stored on Drive (see
                                # GEM_PATH / DATA_PATH in the next cells)

Load GEM

In [ ]:
GEM_PATH = "/content/drive/MyDrive/Human-GEM.xml"
project_folder = "/content/drive/MyDrive/GEM Results/Ker1/"

model = cobra.io.read_sbml_model(GEM_PATH)
model.solver = "hybrid"

print("Model loaded:", model.name)
print("Reactions:",    len(model.reactions))
print("Metabolites:",  len(model.metabolites))
print("Genes:",        len(model.genes))
print(f"Solver: {model.solver.interface.__name__}")

# Baseline FBA, used only as check
baseline_solution = model.optimize()
BASELINE_BIOMASS = baseline_solution.objective_value
print(f"Baseline biomass (unconstrained): {BASELINE_BIOMASS:.4f}")
# Sanity check only - confirms Human-GEM can grow with no applied constraints
# before any expression/metabolomics constraints are layered on later. Not
# used anywhere downstream.

Load Input Data

In [ ]:
DATA_PATH = "/content/drive/MyDrive/gem_input_data7.pkl"

with open(DATA_PATH, "rb") as f:
    gem_data = pickle.load(f)

metadata        = gem_data["metadata"]
nmr             = gem_data["nmr"]
microbiome      = gem_data["microbiome"]
pathways        = gem_data["pathways"]
mean_expr       = gem_data["mean_expr"]
mean_expr_high  = gem_data["mean_expr_high"]
mean_expr_low   = gem_data["mean_expr_low"]
metabolite_map  = gem_data["metabolite_map"]
gem_metabolites = gem_data["gem_metabolites"]
common_ids      = gem_data["common_ids"]

print("Loaded GEM input data.")
print("Samples:",                    len(common_ids))
print("Metabolites for constraints:", gem_metabolites)
print("Metadata shape:",  metadata.shape)
print("NMR shape:",        nmr.shape)
print("Mean expr shape:",  mean_expr.shape)

# Verify the mapping integration
print("Mapped Metabolites to Model IDs:")
for name, mam_id in gem_data['metabolite_map'].items():
    try:
        # Check if the ID exists in loaded model
        met = model.metabolites.get_by_id(mam_id)
        print(f"Verified: {name} as {mam_id}")
    except KeyError:
        print(f"Warning: {mam_id} ({name}) not found in loaded model.")
# Catches mismatches between the metabolite→MAR-ID mapping built during
# GEM_Data_Loading and the actual Human-GEM version loaded here - these two
# can drift out of sync if either the model or the mapping file is updated
# independently.

Exchange Reaction ID Map for Constraining Models

In [ ]:
# Define directions: Most things are taken up; byproduct/waste are secreted.
# If a metabolite is a known secreted product (e.g. lactate, urea, creatinine),
# its exchange bound should allow secretion (positive/outward flux) rather
# than uptake - see METABOLITE_DIRECTION (defined below) for the actual
# per-metabolite direction lookup used when applying constraints.


# Human_GEM Exchange Reaction IDs - for applying uptake/secretion constraints
# Exchange mapping dictionary for each metabolite and its corresponding Human_GEM ID
# Use these IDs to add constraints
exchange_map = {
    "Glucose": "MAR09034", "Arginine": "MAR09066", "Urea": "MAR09081",
    "Lactic acid": "MAR09135", "Succinic acid": "MAR09415", "Proline": "MAR09068",
    "Lysine": "MAR09041", "Alanine": "MAR09061", "Tyrosine": "MAR09064",
    "Isoleucine": "MAR09039", "Tryptophan": "MAR09045", "Histidine": "MAR09040",
    "Ammonia": "MAR09074", "Protons": "MAR09048",

    "Valine": "MAR09046", "Leucine": "MAR09042", "Phenylalanine": "MAR09043",
    "Threonine": "MAR09044", "Methionine": "MAR09047", "Serine": "MAR09069",
    "Glycine": "MAR09065", "Aspartic acid": "MAR09062", "Glutamic acid": "MAR09063",
    "Asparagine": "MAR09067", "Glutamine": "MAR09070", "Cysteine": "MAR09071",
    "Ornithine": "MAR09080", "Citrulline": "MAR09078",
    "Pyroglutamic acid": "MAR09072",
    "Formic acid":      "MAR09126",
    "Hypoxanthine?":    "MAR09241",
    "Urocanic acid":    "MAR09082",
    "Maleic acid":      "MAR09136",
    "Pimelic (or suberic) acid": "MAR09121",

    "Citric acid": "MAR09124", "Fumaric acid": "MAR09127", "Malic acid": "MAR09137",
    "Pyruvic acid": "MAR09140", "Acetoacetic acid": "MAR09109", "Acetic acid": "MAR09123",
    "Creatinine": "MAR09075", "Creatine": "MAR09076", "Choline": "MAR09260",
    "Taurine": "MAR09073", "myo-Inositol": "MAR09231", "Glycerol": "MAR09132"
}

# Define directions: Most things are taken up; byproduct/waste are secreted.
# If a metabolite isn't here, it defaults to 'uptake' in the logic below.
METABOLITE_DIRECTION = {
    "Lactic acid": "secreted",
    "Succinic acid": "secreted",
    "Urea": "secreted",
    "Ammonia": "secreted",
    "Creatinine": "secreted",
    "Acetic acid": "secreted",
    "Protons": "secreted"
}

# Remove the 'Unknowns' and contaminants to clean up the warnings
to_remove = ["Unknown 1", "Unknown 2", "Unknown 3", "Unknown 4", "Epsilon-Caprolactam"]
for item in to_remove:
    if item in exchange_map:
        del exchange_map[item]

print(f"Total mapped metabolites: {len(exchange_map)}")



def get_exchange_id(model, mam_id):
    """Finds the true exchange/boundary reaction for a metabolite MAM ID."""
    try:
        met = model.metabolites.get_by_id(mam_id)
    except KeyError:
        return None
    exchanges = [r for r in met.reactions if len(r.metabolites) == 1]
    return exchanges[0].id if exchanges else None

for nmr_name, mam_id in gem_data['metabolite_map'].items():
    if nmr_name in exchange_map:
        continue
    ex_id = get_exchange_id(model, mam_id)
    if ex_id:
        exchange_map[nmr_name] = ex_id
        print(f"Linked: {nmr_name} → {ex_id}")
    else:
        print(f"No exchange found for: {nmr_name} ({mam_id})")

def find_exchange_reaction(model, metabolite_name):
    """Searches for an exchange reaction matching the metabolite name."""
    for met in model.metabolites:
        # Check if metabolite name matches NMR name (case-insensitive)
        if metabolite_name.lower() in met.name.lower():
            # Find the exchange reaction for this metabolite
            exchanges = [r for r in met.reactions if len(r.metabolites) == 1]
            if exchanges:
                return exchanges[0].id
    return None

# Update map based on your specific NMR columns
for met_name in nmr.columns:
    if met_name not in exchange_map:
        found_id = find_exchange_reaction(model, met_name)
        if found_id:
            exchange_map[met_name] = found_id
            print(f"Linked (Fuzzy): {met_name} -> {found_id}")
        else:
            print(f"Warning: Could not find ID for {met_name}")

print(len(exchange_map))
print(len(gem_data['metabolite_map']))
print(exchange_map)

Map Gene Symbols to Ensembl IDs

In [ ]:
mg = mygene.MyGeneInfo()
symbols = list(mean_expr.index)
mapping = mg.querymany(symbols, scopes='symbol', fields='ensembl.gene', species='human')
symbol_to_ens = {}
for entry in mapping:
    if 'notfound' in entry and entry['notfound']:
        continue # gene symbol not recognised by mygene - dropped,
                   # so any downstream expression lookup for it returns nothing
    if 'ensembl' not in entry:
        continue
    ens = entry['ensembl']
    if isinstance(ens, list):
        ens = ens[0] # some symbols map to multiple Ensembl IDs (e.g. paralogs/
                       # historical aliases) - takes the first

    if isinstance(ens, dict) and 'gene' in ens:
        symbol_to_ens[entry['query']] = ens['gene']

print(f"Mapped {len(symbol_to_ens)} gene symbols to Ensembl IDs.")

Gene ID Conversion (used to map sample-level expression to Ensembl)

In [ ]:
def convert_expr_to_ensembl(expr_df, mapping_dict):
  # Reindex an expression Series/DataFrame from gene symbols to Ensembl IDs
    #using mapping_dict, dropping any genes with no mapping. Where multiple
    #symbols map to the same Ensembl ID, their expression values are averaged

    expr_df = expr_df.loc[expr_df.index.intersection(mapping_dict.keys())].copy()
    expr_df.index = expr_df.index.map(mapping_dict)
    expr_df = expr_df.groupby(expr_df.index).mean()
    return expr_df

GPR_Aware GIMME Weighting

In [ ]:
def evaluate_gpr(gpr_node, expr_lookup):
    """
    Recursively evaluate a COBRApy GPR expression tree.
    AND nodes -> min (enzyme complex: all subunits needed)
    OR  nodes -> max (isozymes: any isoform sufficient)
    """
    if gpr_node is None:
        return None

    if hasattr(gpr_node, 'id'):
        return expr_lookup.get(gpr_node.id, None)
    if hasattr(gpr_node, 'name'):
        return expr_lookup.get(gpr_node.name, None)

    if hasattr(gpr_node, 'args') and isinstance(gpr_node, ast.BoolOp) and isinstance(gpr_node.op, ast.And):
        child_vals = [evaluate_gpr(child, expr_lookup) for child in gpr_node.args]
        child_vals = [v for v in child_vals if v is not None]
        return min(child_vals) if child_vals else None

    if hasattr(gpr_node, 'args') and isinstance(gpr_node, ast.BoolOp) and isinstance(gpr_node.op, ast.Or):
        child_vals = [evaluate_gpr(child, expr_lookup) for child in gpr_node.args]
        child_vals = [v for v in child_vals if v is not None]
        return max(child_vals) if child_vals else None

    if hasattr(gpr_node, 'body'):
        return evaluate_gpr(gpr_node.body, expr_lookup)

    return None


def build_reaction_weights(model, expr_series, threshold=None):
    """
    weight = max(0, threshold - gpr_expression)
    0.0  -> reaction supported by expression (no GIMME penalty)
    >0.0 -> penalised proportionally to expression deficit
    """
    nonzero_expr = expr_series[expr_series > 0]
    if threshold is None:
        threshold = float(np.percentile(nonzero_expr, 10)) if len(nonzero_expr) > 0 else 0.1

    expr_lookup = expr_series.to_dict()
    weights = {}

    for rxn in model.reactions:
        if not rxn.gene_reaction_rule or rxn.gene_reaction_rule.strip() == '':
            weights[rxn.id] = 0.0
            continue

        try:
            tree = ast.parse(rxn.gene_reaction_rule, mode='eval')
            rxn_expr = evaluate_gpr(tree, expr_lookup)
        except Exception:
            rxn_expr = None

        if rxn_expr is None:
            weights[rxn.id] = float(threshold)
        else:
            weights[rxn.id] = max(0.0, threshold - float(rxn_expr))

    return weights, threshold


Cleanup Leftover Demand Reactions on Base Model

In [ ]:
# DM_ (demand) reactions are leftover artifacts from the base Human-GEM model
# not used in this pipeline's constraint-application logic - removing them
# up front avoids them being included in downstream reaction counts
# or FVA loops.

dms_to_remove = [r.id for r in model.reactions if r.id.startswith("DM_")]
if dms_to_remove:
    model.remove_reactions(dms_to_remove)
    print(f"Cleaned {len(dms_to_remove)} leftover demand reactions.")


Medium/Metabolite Constraints

In [ ]:
ESSENTIAL_OPEN_EXCHANGES = {
    "MAR09048": (-1000.0, 1000.0),  # Protons
    "MAR09150": (-1000.0, 1000.0),  # H2O
    "MAR09155": (-1000.0, 1000.0),  # O2
    "MAR09079": (-1000.0, 1000.0),  # Inorganic phosphate (Pi)
    "MAR09153": (-1000.0, 1000.0),  # CO2
    "MAR09143": (-1000.0, 1000.0),  # Sulphate
    "MAR09074": (-1000.0, 1000.0),  # Ammonia
    "MAR09047": (-1.0, 1000.0),     # Methionine
    "MAR09065": (-1.0, 1000.0),     # Glycine
    "MAR09070": (-1.0, 1000.0),     # Glutamine
    "MAR09071": (-1.0, 1000.0),     # Cysteine
    "MAR09147": (-1.0, 1000.0),     # Magnesium
    "MAR09145": (-1.0, 1000.0),     # Potassium
    "MAR09141": (-1.0, 1000.0),     # Calcium
    "MAR09146": (-1.0, 1000.0),     # Sodium
    "MAR09144": (-1.0, 1000.0),     # Chloride
    "MAR09142": (-0.01, 1000.0),    # Iron (Fe2+)
    "MAR09149": (-0.01, 1000.0),    # Manganese
    "MAR09148": (-0.01, 1000.0),    # Cu2+
    "MAR09151": (-0.01, 1000.0),    # Zn2+
    "MAR09100": (-0.01, 1000.0),    # Biotin
    "MAR09107": (-0.01, 1000.0),    # Pantothenate
    "MAR09101": (-0.01, 1000.0),    # Folate (B9)
    "MAR09108": (-0.01, 1000.0),    # Riboflavin (B2)
    "MAR09105": (-0.01, 1000.0),    # Nicotinate (B3)
    "MAR09104": (-0.01, 1000.0),    # Pyridoxine (B6)
    "MAR09109": (-0.01, 1000.0),    # Thiamine (B1)
    "MAR09132": (-0.1, 1000.0),     # Glycerol
    "MAR09260": (-0.1, 1000.0),     # Choline
    "MAR09231": (-0.1, 1000.0),     # myo-Inositol
    "MAR09228": (-0.01, 1000.0),    # Cholesterol
    "MAR09115": (-0.01, 1000.0),    # Linoleic acid
    "MAR09103": (-0.01, 1000.0),    # Vitamin B12
    "MAR09083": (-1000.0, 1000.0),  # Bicarbonate
}

TRICKLE_VALUE = -0.01  # Small "background" uptake allowed for metabolites with no direct
                        # NMR measurement, so the model isn't forced to zero-flux on
                        # everything we didn't explicitly measure - avoids unrealistic
                        # infeasibility from over-constraining.


def medium_relaxed(model_copy):
    n_relaxed = 0
    for rxn in model_copy.boundary:
        if rxn.lower_bound < 0:
            rxn.lower_bound = TRICKLE_VALUE
            n_relaxed += 1
    return n_relaxed


def apply_metabolite_constraints(base_model, nmr_profile, exchange_map,
                                  metabolite_direction, scale=None):
    if scale is None:
        scale = SCALE

    m = base_model.copy()

    to_remove = [r.id for r in m.reactions if r.id.startswith("DM_") or r.id.startswith("TEST_")]
    if to_remove:
        m.remove_reactions(to_remove)

    n_relaxed = medium_relaxed(m)
    print(f"  Relaxed {n_relaxed} exchange reactions (trickle set to {TRICKLE_VALUE}).")

    for rxn_id, (lb, ub) in ESSENTIAL_OPEN_EXCHANGES.items():
        try:
            rxn = m.reactions.get_by_id(rxn_id)
            rxn.lower_bound = lb
            rxn.upper_bound = ub
        except KeyError:
            continue

    for met_name, rxn_id in exchange_map.items():
        if met_name not in nmr_profile.index:
            continue
        level = float(nmr_profile[met_name])
        level = max(0.0, level)
        flux = scale * level
        rxn = m.reactions.get_by_id(rxn_id)
        direction = metabolite_direction.get(met_name, "uptake")

        if direction == "uptake":
            rxn.lower_bound = -flux
        else:
            rxn.upper_bound = flux
            rxn.lower_bound = 0.0

    return m



Skin pH Context

In [ ]:
def apply_environmental_context(model, condition_type):
    """
    High SA -> pH 6.13 (more alkaline) -> easier proton efflux
    Low SA  -> pH 5.13 (more acidic)   -> harder proton efflux
    """
    m = model.copy()
    h_plus_exchange = "MAR09048"
    if condition_type == 'high':
        m.reactions.get_by_id(h_plus_exchange).lower_bound = -1000.0 # unrestricted
                                                                        # proton efflux
    elif condition_type == 'low':
        m.reactions.get_by_id(h_plus_exchange).lower_bound = -400.0 # more
                                                                        # restricted,
                                                                        # reflecting
                                                                        # acidic skin pH
    return m
    # NOTE: the pH figures (6.13 vs 5.13)
    # came from measured skin pH per group

Metabolic Task Validation

In [ ]:
METABOLIC_TASKS_BY_CT = {
    "KER1": [
        ("Biomass production",          "MAR13082", 0.01),
        ("Glucose uptake",               "MAR09034", None),
        ("ATP production (via ATPM)",    "MAR03964", 0.01),
        ("Fatty acid synthesis",         "MAR00010", None),
        ("Amino acid catabolism (Arg)",  "MAR04209", None),
    ],
    "T CELL": [
        ("Biomass production (Proliferation)", "MAR13082", 0.01),
        ("Glucose uptake (Glycolytic engine)",  "MAR09034", None),
        ("ATP production (via ATPM)",           "MAR03964", 0.01),
        ("Lactate secretion (Aerobic Glyco)",   "MAR09135", None),
        ("Glutamine uptake (Glutaminolysis)",   "MAR09070", None),
        ("TCA Cycle Capacity (SDH reaction)",   "MAR04360", None),
    ],
}


def run_metabolic_tasks(constrained_model, condition_label, tasks):
  # Check that a set of core metabolic functions (biomass,
   # glucose uptake, ATP production, etc.) are still achievable under the
   # current constraints - sanity check before GIMME/E-Flux, so a
   # broken/over-constrained model is caught early

    print(f"\n--- Metabolic Task Validation: {condition_label} ---")
    all_passed = True
    for task_name, rxn_id, min_flux in tasks:
        try:
            rxn = constrained_model.reactions.get_by_id(rxn_id)
        except KeyError:
            print(f"  [SKIP] {task_name:<40} -- reaction {rxn_id} not in model")
            continue

        with constrained_model as test_model:
            test_model.objective = rxn_id
            sol = test_model.optimize()
            achieved = sol.objective_value if sol.status == 'optimal' else 0.0

        passed = (achieved >= min_flux) if min_flux is not None else (achieved > 1e-9)
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_passed = False
        print(f"  [{status}] {task_name:<40} max flux = {achieved:.4f}")

    print("  All tasks passed." if all_passed else "  One or more tasks FAILED.")
    return all_passed

GIMME

In [ ]:
def run_gimme(model, reaction_weights, fraction_of_optimum=0.9):
    rxn_map = {r.id: r for r in model.reactions}

    # COBRApy stores the objective as optlang variables, not a Reaction object directly,
    # so recover the biomass reaction by checking which reaction's forward/reverse
    # variable has a nonzero coefficient in the current objective.

    coeffs = model.objective.get_linear_coefficients(model.variables)
    biomass_rxn = next((r for r in model.reactions
                        if r.forward_variable in coeffs or r.reverse_variable in coeffs), None)
    if not biomass_rxn:
        raise ValueError("Could not identify the objective reaction.")

    original_objective = model.objective
    original_lb = biomass_rxn.lower_bound

    try:
      #find the maximum achievable biomass under current constraints
        fba_sol = model.optimize()
        if fba_sol.status != 'optimal':
            return None, None

        max_biomass = fba_sol.objective_value
        # lock in growth at 90% of max (fraction_of_optimum) so GIMME can't
        # sacrifice growth entirely while minimising penalised flux - keeps the
        # model biologically realistic rather

        biomass_rxn.lower_bound = max_biomass * fraction_of_optimum
        #build a new objective that penalises flux through reactions whose
        # expression support is low (weight > 0), pushing flux toward
        # expression-consistent pathways.
        objective_dict = {}
        for key, weight in reaction_weights.items():
            r_id = key.id if hasattr(key, 'id') else str(key)
            if weight > 0 and r_id in rxn_map:
                rxn = rxn_map[r_id]
                objective_dict[rxn.forward_variable] = weight
                objective_dict[rxn.reverse_variable] = weight

        model.objective = model.problem.Objective(Zero, direction="min")
        model.objective.set_linear_coefficients(objective_dict)

        gimme_sol = model.optimize()
        if gimme_sol.status != 'optimal':
            print(f"  WARNING: GIMME-weighted solve status='{gimme_sol.status}' -- treating as failed.")
            return None, max_biomass

        return gimme_sol, max_biomass

    finally:

        # Always restore the model's original state, even if solving fails above -
        # otherwise later cells would reuse a mutated objective/bound.

        biomass_rxn.lower_bound = original_lb
        model.objective = original_objective

Post-Processing Functions

In [ ]:
def open_nutrient_doors(model):
  # Relax any fully-closed exchange reaction to allow a small amount of uptake (-10),
    # so the model isn't artificially starved of unmeasured nutrients before task validation.

    for reaction in model.exchanges:
        if reaction.lower_bound == 0:
            reaction.lower_bound = -10.0
    print(f"Doors opened for {model.id}")


def cleanup(model):
  # Rebuild the model without duplicate reaction IDs or leftover demand (DM_) reactions,
   # then sanity-check that it can still grow. Returns a fresh cobra.Model.

    seen_ids = set()
    to_keep = []
    removed_duplicates = 0
    removed_dms = 0
    for rxn in model.reactions:
        if rxn.id.startswith("DM_"):
            removed_dms += 1
            continue
        if rxn.id in seen_ids:
            removed_duplicates += 1
            continue
        seen_ids.add(rxn.id)
        to_keep.append(rxn)

    new_model = cobra.Model(model.id)
    new_model.add_reactions(to_keep)

    if isinstance(model.objective, cobra.Reaction):
        new_model.objective = model.objective.id
    else:
        new_model.objective = model.objective

    print(f"  Removed {removed_dms} DM reactions.")
    print(f"  Removed {removed_duplicates} duplicate IDs.")

    new_model.solver = 'hybrid' # 'hybrid' chosen for stability on this model size -
                                  # falls back between solvers if one struggles to converge

    growth = new_model.slim_optimize()
    if growth <= 0:
        print(f"  WARNING: Model is non-functional after cleanup! (Growth: {growth:.4f})")
    else:
        print(f"  Model healthy! Growth: {growth:.4f}")
    return new_model


def run_safe_fva(model, reaction_list, fixed_growth_rate):
    #Run flux variability analysis with growth pinned to a fixed rate
    #so flux ranges reflect variability at a specific, comparable growth state
    #important when comparing High-SA vs Low-SA models that may have different max growth

    results = {}
    total = len(reaction_list)
    print(f"Starting Safe FVA on {total} reactions at fixed growth {fixed_growth_rate}...")

    coeffs = model.objective.get_linear_coefficients(model.variables)
    biomass_rxn = next((r for r in model.reactions
                        if r.forward_variable in coeffs or r.reverse_variable in coeffs), None)
    if not biomass_rxn:
        raise ValueError("Biomass reaction not found in objective.")

    with model:
      # context manager: bound changes below are reverted automatically on exit

        biomass_rxn.lower_bound = fixed_growth_rate
        biomass_rxn.upper_bound = fixed_growth_rate

        rxn_map = {rid: model.reactions.get_by_id(rid) for rid in reaction_list if rid in model.reactions}
        for i, rid in enumerate(rxn_map.keys()):
            rxn = rxn_map[rid]
            model.objective = rxn
            model.objective_direction = 'min'
            f_min = model.slim_optimize()
            model.objective_direction = 'max'
            f_max = model.slim_optimize()
            results[rid] = {"minimum": f_min, "maximum": f_max}
            if (i + 1) % 100 == 0:
                print(f"  Processed {i+1}/{len(rxn_map)}...")

    return pd.DataFrame(results).T

In [ ]:
#The scale factor is computed from the global mean glucose across all 6 samples, then applied when building each group model
metadata_subset = gem_data["metadata"]
nmr_subset      = gem_data["nmr"]


# NMR -> flux scale factor, calibrated on the glucose mean across ALL
# 6 samples
TARGET_GLUCOSE_FLUX_BY_CT = {
    "KER1":   1.0,
    "T CELL": 10.0,   # higher baseline glycolytic capacity of activated T cells
}
nmr_all_mean = nmr_subset.mean()

SCALE_BY_CT = {}
for ct, target in TARGET_GLUCOSE_FLUX_BY_CT.items():
    glucose_nmr_mean = float(nmr_all_mean["Glucose"])  # nmr_all_mean already computed earlier
    SCALE_BY_CT[ct] = target / glucose_nmr_mean if glucose_nmr_mean > 0 else 0.01
    print(f"[{ct}] SCALE = {SCALE_BY_CT[ct]:.6f} (target glucose flux = {target})")


Pooled Models

In [ ]:
# POOLED: average inputs per (SA_status × cell type),
# E-Flux + pFBA pipeline on 2 models per cell type

PATIENT_COL = "eg_id"
CELL_TYPES  = ["KER1", "T CELL"]
BIOMASS_ID  = "MAR13082"
MIN_CELLS   = 20

metadata_subset = gem_data["metadata"]
nmr_subset      = gem_data["nmr"]
sample_info = metadata_subset.loc[list(common_ids), ["SA_status", PATIENT_COL]].copy()
sample_info = metadata_subset.loc[list(common_ids), ["SA_status", PATIENT_COL]].copy()
GROUPS = ["High", "Low"]



# E-Flux: scale each reaction's bounds by its (GPR-resolved) expression.
# High expression -> wide bounds; low expression -> narrow (but never
# hard-zero) bounds. Never imposes a separate minimisation objective,
# so it does not conflict with the biomass lock

def reaction_expression(model, expr_series):
    """GPR-resolved expression per reaction (reuses evaluate_gpr).
    Returns dict {rxn_id: expr_value or None}."""
    expr_lookup = expr_series.to_dict()
    rxn_expr = {}
    for rxn in model.reactions:
        if not rxn.gene_reaction_rule or rxn.gene_reaction_rule.strip() == "":
            rxn_expr[rxn.id] = None          # no GPR -> leave bounds untouched
            continue
        try:
            tree = ast.parse(rxn.gene_reaction_rule, mode="eval")
            rxn_expr[rxn.id] = evaluate_gpr(tree, expr_lookup)
        except Exception:
            rxn_expr[rxn.id] = None
    return rxn_expr


def apply_eflux(m, expr_series, biomass_id, protected_ids=None):
    """Scale internal reaction bounds by normalised expression.

    - Reactions with no GPR or unmapped genes: bounds left as-is.
    - Reactions with expression: bounds scaled to expr/max_expr * original.
    - Exchange/boundary reactions and the biomass reaction are NOT scaled
      (their bounds come from NMR/pH constraints and the growth lock).
    """
    if protected_ids is None:
        protected_ids = set()

    rxn_expr = reaction_expression(m, expr_series)
    vals = [v for v in rxn_expr.values() if v is not None and v > 0]
    if not vals:
        return m, 0
    max_expr = max(vals)

    boundary_ids = {r.id for r in m.boundary}
    n_scaled = 0

    for rxn in m.reactions:
        if rxn.id in boundary_ids or rxn.id == biomass_id or rxn.id in protected_ids:
            continue
        e = rxn_expr[rxn.id]
        if e is None:
            continue                          # untouched: no expression info
        scale = max(e, 0.0) / max_expr        # in [0, 1]
        # Scale bounds toward zero by the expression fraction, but keep a
        # small floor so nothing is fully blocked (preserves feasibility).
        floor = 1e-3
        factor = max(scale, floor)
        if rxn.upper_bound > 0:
            rxn.upper_bound = rxn.upper_bound * factor
        if rxn.lower_bound < 0:
            rxn.lower_bound = rxn.lower_bound * factor
        n_scaled += 1

    return m, n_scaled

mean_expr_by_sample_ens = {}
for ct in CELL_TYPES:
    mean_expr_by_sample_ens[ct] = {}
    counts = gem_data["cell_counts_by_sample"][ct]
    for sid in list(common_ids):
        n_cells = counts.loc[sid, "n_cells"]
        if n_cells < MIN_CELLS:
            print(f"Skipping {sid} for {ct}: only {n_cells} cells (< {MIN_CELLS})")
            continue
        expr_series = convert_expr_to_ensembl(
            gem_data["mean_expr_by_sample"][ct][[sid]].iloc[:, 0], symbol_to_ens
        )
        mean_expr_by_sample_ens[ct][sid] = expr_series

#  Average NMR per group (across that group's samples)
nmr_by_group = {}
for grp in GROUPS:
    sids_grp = sample_info[sample_info["SA_status"] == grp].index.tolist()
    nmr_by_group[grp] = nmr_subset.loc[sids_grp].mean(axis=0)
    print(f"NMR [{grp}]: averaged across {len(sids_grp)} samples")

# Average per-cell-type expression per group (across that group's
#    samples that passed MIN_CELLS for that cell type)
expr_by_group = {ct: {} for ct in CELL_TYPES}
for ct in CELL_TYPES:
    for grp in GROUPS:
        sids_grp = [sid for sid in sample_info[sample_info["SA_status"] == grp].index
                    if sid in mean_expr_by_sample_ens[ct]]
        if not sids_grp:
            print(f"[{ct}/{grp}]: no samples after MIN_CELLS — skipping.")
            continue
        # Stack each sample's per-gene Series into a DataFrame, then mean per gene
        expr_df = pd.DataFrame({sid: mean_expr_by_sample_ens[ct][sid] for sid in sids_grp})
        expr_by_group[ct][grp] = expr_df.mean(axis=1)
        print(f"[{ct}/{grp}]: averaged expression across {len(sids_grp)} samples")

# Build one model per (group × cell type), same constraints as
#    per-sample but using the group-averaged inputs
def build_pooled_model(grp, ct):
    nmr_row = nmr_by_group[grp]
    m = apply_metabolite_constraints(
        model, nmr_row, exchange_map, METABOLITE_DIRECTION,
        scale=SCALE_BY_CT[ct]
    )
    m = apply_environmental_context(m, grp.lower())
    return m

pooled_models = {}
for ct in CELL_TYPES:
    pooled_models[ct] = {}
    for grp in GROUPS:
        if grp not in expr_by_group[ct]:
            continue
        m = build_pooled_model(grp, ct)
        pooled_models[ct][grp] = m
        sol = m.optimize()
        bio = sol.objective_value if sol.status == "optimal" else 0.0
        print(f"[{ct}/{grp}]: status={sol.status}, biomass={bio:.4f}")

# Shared growth rate from POST-E-FLUX-SCALING max biomass
#    (same logic as per-sample, but only 2 models per cell type)
shared_growth_rate = {}
for ct in CELL_TYPES:
    scaled_max = {}
    for grp in pooled_models[ct]:
        m = pooled_models[ct][grp].copy()
        m, _ = apply_eflux(m, expr_by_group[ct][grp], biomass_id=BIOMASS_ID)
        scaled_max[grp] = m.slim_optimize()
    shared_growth_rate[ct] = min(scaled_max.values()) * 0.7
    print(f"[{ct}] Scaled max biomass per group: {scaled_max}")
    print(f"[{ct}] Shared growth rate (70% of scaled min): {shared_growth_rate[ct]:.4f}")

#  E-Flux + locked-biomass pFBA per (group × cell type)
eflux = {}
for ct in CELL_TYPES:
    eflux[ct] = {}
    for grp in pooled_models[ct]:
        m = pooled_models[ct][grp].copy()
        m, n_scaled = apply_eflux(m, expr_by_group[ct][grp], biomass_id=BIOMASS_ID)
        biomass_rxn = m.reactions.get_by_id(BIOMASS_ID)
        biomass_rxn.lower_bound = shared_growth_rate[ct]
        biomass_rxn.upper_bound = shared_growth_rate[ct]
        try:
            sol = pfba(m)
            eflux[ct][grp] = sol
            print(f"[{ct}/{grp}]: OK (scaled {n_scaled} reactions, "
                  f"pFBA obj={sol.objective_value:.4f})")
        except Exception as e:
            eflux[ct][grp] = None
            print(f"[{ct}/{grp}]: FAILED ({e})")

#  Flux matrix (2 columns per cell type)
flux_matrix = {}
summary = {}
for ct in CELL_TYPES:
    grps_ok = [g for g in eflux[ct] if eflux[ct][g] is not None]
    fm = pd.DataFrame({grp: eflux[ct][grp].fluxes for grp in grps_ok})
    flux_matrix[ct] = fm
    print(f"[{ct}] fm shape: {fm.shape}")

    if "High" in fm.columns and "Low" in fm.columns:
        s = pd.DataFrame({
            "high_flux": fm["High"],
            "low_flux":  fm["Low"],
            "group_diff": fm["High"] - fm["Low"],
        })
        summary[ct] = s

Saving

In [ ]:
#  EXPORT: pooled (group × cell-type) GEM results
#
# Two models per cell type (High, Low), each from group-averaged
# inputs.
pooled_dir = f"{project_folder}Pooled_PerCellType/"
os.makedirs(pooled_dir, exist_ok=True)

for ct in CELL_TYPES:
    if ct not in flux_matrix:
        continue
    fm = flux_matrix[ct]
    ct_tag = ct.replace(" ", "")

    # Flux matrix: reactions × 2 groups (columns are just
    #    "High" and "Low", already plain strings in pooled fm)
    fm.to_csv(f"{pooled_dir}FluxMatrix_{ct_tag}.csv")

    #  Summary table (high_flux, low_flux, group_diff) with
    #    reaction metadata attached
    s = summary[ct].copy()
    rxn_meta = {r.id: (r.name, r.subsystem) for r in model.reactions}
    s[["name", "subsystem"]] = s.index.to_series().apply(
        lambda x: pd.Series(rxn_meta.get(x, ("Unknown", "Unknown")))
    )
    s.to_csv(f"{pooled_dir}Summary_{ct_tag}.csv")

    print(f"[{ct}] Saved FluxMatrix and Summary to {pooled_dir}")

print(f"\nAll pooled (group × cell type) results saved under:\n  {pooled_dir}")
print("Note: this pooled comparison averages 4 High samples (2 patients) "
      "vs 2 Low samples (1 patient) — High vs Low is confounded with "
      "patient identity in the Low group. Per-sample results remain in "
      f"{project_folder}PerSample_PerCellType/.")

Visualisation

In [ ]:
# VISUALISE: pooled (group × cell type) GEM results

project_folder = "/content/drive/MyDrive/GEM Results/Ker1/"
pooled_dir = f"{project_folder}Pooled_PerCellType/"
plot_dir   = f"{pooled_dir}Plots/"
os.makedirs(plot_dir, exist_ok=True)

CELL_TYPES = ["KER1", "T CELL"]

# Human-readable overrides for reaction/pathway labels used in plots - covers
# MAR IDs where the model's own `name` field was missing/NaN, and cases where
# the model's systematic enzyme name (e.g. "citrate hydro-lyase...") is too
# technical to be legible on an axis label.

NAME_OVERRIDES = {
    # MAR IDs where summary stored NaN
    "MAR05043": "Mitochondrial phosphate carrier",
    "MAR04964": "Mitochondrial citrate/malate exchanger",
    "MAR04855": "Mitochondrial fumarate/malate exchanger",
    "MAR03016": "Eicosanoid metabolism",
    "MAR01322": "Prostaglandin biosynthesis 1",
    "MAR01323": "Prostaglandin biosynthesis 2",
    "MAR01324": "Prostaglandin biosynthesis 3",
    "MAR05998": "TCA cycle reaction (MAR05998)",
    "MAR06033": "TCA cycle reaction (MAR06033)",
    "MAR04919": "TCA cycle reaction (MAR04919)",
    "MAR06413": "TCA succinylation reaction",
    "MAR04209": "TCA dicarboxylate reaction",
    "MAR01321": "Prostaglandin E2 synthase",
    "MAR03820": "Pyrroline-5-carboxylate reductase",
    "MAR06918": "NADH dehydrogenase (Complex I)",
    # Readable name overrides
    "citrate hydro-lyase (cis-aconitate-forming)":          "Aconitase (citrate -> aconitate)",
    "isocitrate hydro-lyase (cis-aconitate-forming)":       "Aconitase (isocitrate -> aconitate)",
    "citrate hydroxymutase":                                "Citrate isomerase",
    "(S)-malate hydro-lyase (fumarate-forming)":            "Fumarase (malate -> fumarate)",
    "carbonate hydro-lyase (carbon-dioxide-forming)":       "Carbonic anhydrase",
    "Exchange of HCO3-":                                    "Bicarbonate exchange",
    "Exchange of Pi":                                       "Inorganic phosphate exchange",
    "IMP:NAD+ oxidoreductase":                              "IMP dehydrogenase",
    "xanthosine:orthophosphate ribosyltransferase":         "Purine nucleoside phosphorylase (xanthosine)",
    "xanthosine 5'-phosphate phosphohydrolase":             "XMP phosphatase",
    "5,10-methylenetetrahydrofolate:NAD+ oxidoreductase":   "Methylenetetrahydrofolate dehydrogenase",
    "ATP:ADP phosphatransferase":                           "ATP/ADP translocase",
    "succinyl-CoA:enzyme N6-(dihydrolipoyl)lysine S-succinyltransferase": "Dihydrolipoamide succinyltransferase (OGDH complex)",
    "(S)-Lactate:NAD+ oxidoreductase":                      "Lactate dehydrogenase",
    "D-phosphoglycerate 2,3-phosphomutase":                 "Phosphoglycerate mutase",
    "L-proline:(acceptor) oxidoreductase":                  "Proline oxidase",
    "L-glutamate gamma-semialdehyde:NAD+ oxidoreductase":   "Glutamate semialdehyde dehydrogenase",
    "(S)-1-pyrroline-5-carboxylate:NAD+ oxidoreductase":    "Pyrroline carboxylate reductase",
    "L-Aspartate:2-oxoglutarate aminotransferase":          "Aspartate aminotransferase",
    "ferrocytochrome-c:oxygen oxidoreductase":              "Cytochrome c oxidase (Complex IV)",
    "(S)-malate:NAD+ oxidoreductase":                       "Malate dehydrogenase",
}

def reaction_label(rxn_id, summary_df):

    # Return the most readable available label for a reaction:
    # override if one exists, else the model's own name (unless that name is
    # itself overridden), else fall back to the raw reaction ID

    if rxn_id in NAME_OVERRIDES:
        return NAME_OVERRIDES[rxn_id]
    if rxn_id in summary_df.index:
        n = summary_df.loc[rxn_id, "name"]
        if pd.notna(n) and n not in ("Unknown", ""):
            if n in NAME_OVERRIDES:
                return NAME_OVERRIDES[n]
            return n
    return rxn_id



# Reload CSVs from disk
# so this visualisation cell can be re-run independently after the main
# pipeline has already saved its output.
flux_matrix = {}
summary     = {}

for ct in CELL_TYPES:
    ct_tag = ct.replace(" ", "")
    fm_path = f"{pooled_dir}FluxMatrix_{ct_tag}.csv"
    su_path = f"{pooled_dir}Summary_{ct_tag}.csv"

    if not os.path.exists(fm_path):
        print(f"[{ct}] FluxMatrix not found at {fm_path}, skipping.")
        continue

    flux_matrix[ct] = pd.read_csv(fm_path, index_col=0)
    summary[ct]     = pd.read_csv(su_path, index_col=0)
    print(f"[{ct}] Loaded: {flux_matrix[ct].shape[0]} reactions x {flux_matrix[ct].shape[1]} groups")

print("\nSetup complete.\n")


#   heatmap of top reactions by |group_diff|
for ct in CELL_TYPES:
    if ct not in flux_matrix:
        continue
    ct_tag = ct.replace(" ", "")
    fm = flux_matrix[ct]
    su = summary[ct]

    print(f"[{ct}] Generating pooled heatmap...")
    top_rxns = su["group_diff"].abs().sort_values(ascending=False).head(20).index.tolist()
    top_rxns = [r for r in top_rxns if r in fm.index]
    row_labels = [reaction_label(r, su) for r in top_rxns]

    plot_vals = fm.loc[top_rxns].values
    # Signed log transform: compresses the wide dynamic range of flux values
    # for display while preserving direction (uptake vs secretion), a
    # plain log would fail on negative fluxes and a linear scale would let a
    # few large-flux reactions dominate the colour scale.

    plot_log  = np.sign(plot_vals) * np.log1p(np.abs(plot_vals))

    fig, ax = plt.subplots(figsize=(5, 9))
    sns.heatmap(
        plot_log,
        xticklabels=list(fm.columns),     #["High", "Low"]
        yticklabels=row_labels,
        cmap="RdBu_r", center=0,
        cbar_kws={"label": "Flux (signed log-scale, mmol gDW^-1 h^-1)"},
        ax=ax,
    )
    ax.set_title(
        f"[{ct}] Pooled flux: top 20 reactions\nby |High - Low| difference"
    )
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.subplots_adjust(left=0.45, right=0.85, bottom=0.10, top=0.92)
    plt.savefig(f"{plot_dir}Heatmap_{ct_tag}.png", dpi=300)
    plt.close("all")
    print(f"[{ct}] Heatmap saved to {plot_dir}Heatmap_{ct_tag}.png")


#  horizontal barplot of top group_diff reactions, coloured by direction.

TOP_N_BAR = 20
MIN_DIFF  = 1.0  # ignore reactions with trivially small group differences
                  # (filters out noise from near-zero flux shifts)


for ct in CELL_TYPES:
    if ct not in flux_matrix:
        continue
    ct_tag = ct.replace(" ", "")
    su = summary[ct]

    print(f"[{ct}] Generating group-difference barplot...")

    # Drop generic/structural subsystems that aren't biologically informative
    # on their own (e.g. "Transport reactions" spans almost the whole model)

    EXCLUDE = {"Miscellaneous", "Pool reactions", "Artificial reactions",
               "Transport reactions", "Exchange/demand reactions"}
    candidates = su[su["group_diff"].abs() > MIN_DIFF].copy()
    if "subsystem" in candidates.columns:
        candidates = candidates[~candidates["subsystem"].isin(EXCLUDE)]
    top = candidates.reindex(
        candidates["group_diff"].abs().sort_values(ascending=False).index
    ).head(TOP_N_BAR)

    if top.empty:
        print(f"[{ct}] No reactions exceed MIN_DIFF={MIN_DIFF}, skipping barplot.")
        continue

    labels = [reaction_label(r, su) for r in top.index]
    colours = ["#c0392b" if d > 0 else "#2980b9" for d in top["group_diff"]]

    fig, ax = plt.subplots(figsize=(8, 0.45 * len(top) + 1.5))
    ax.barh(range(len(top)), top["group_diff"].values, color=colours)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("Group difference in flux\n(High - Low, mmol gDW^-1 h^-1)")
    ax.set_title(f"[{ct}] Pooled comparison: top {len(top)} reactions by |High - Low|")
    plt.tight_layout()
    plt.savefig(f"{plot_dir}GroupDiff_Barplot_{ct_tag}.png", dpi=300)
    plt.close("all")
    print(f"[{ct}] Barplot saved to {plot_dir}GroupDiff_Barplot_{ct_tag}.png")


print(f"\nAll pooled-pipeline plots saved under:\n  {plot_dir}")
print("Reminder: pooled comparison averages 4 High samples (2 patients) vs "
      "2 Low samples (1 patient). High vs Low is confounded with patient "
      "identity in the Low group.")

In [ ]:
print(f"Reactions: {len(model.reactions)}")
print(f"Metabolites: {len(model.metabolites)}")
print(f"Genes: {len(model.genes)}")

In [ ]:
pooled_models = {}
for ct in CELL_TYPES:
    pooled_models[ct] = {}
    for grp in GROUPS:
        if grp not in expr_by_group[ct]:
            continue
        m = build_pooled_model(grp, ct)
        pooled_models[ct][grp] = m
        sol = m.optimize()
        bio = sol.objective_value if sol.status == "optimal" else 0.0
        print(f"[{ct}/{grp}]: status={sol.status}, biomass={bio:.4f}")

In [ ]:
m_test = pooled_models["KER1"]["High"].copy()
_, n_scaled = apply_eflux(m_test, expr_by_group["KER1"]["High"], biomass_id=BIOMASS_ID)
print(f"KER1/High: {n_scaled}")

m_test = pooled_models["KER1"]["Low"].copy()
_, n_scaled = apply_eflux(m_test, expr_by_group["KER1"]["Low"], biomass_id=BIOMASS_ID)
print(f"KER1/Low: {n_scaled}")

m_test = pooled_models["T CELL"]["High"].copy()
_, n_scaled = apply_eflux(m_test, expr_by_group["T CELL"]["High"], biomass_id=BIOMASS_ID)
print(f"T CELL/High: {n_scaled}")

m_test = pooled_models["T CELL"]["Low"].copy()
_, n_scaled = apply_eflux(m_test, expr_by_group["T CELL"]["Low"], biomass_id=BIOMASS_ID)
print(f"T CELL/Low: {n_scaled}")

In [ ]:
shared_growth_rate = {}
for ct in CELL_TYPES:
    scaled_max = {}
    for grp in pooled_models[ct]:
        m = pooled_models[ct][grp].copy()
        m, n_scaled = apply_eflux(m, expr_by_group[ct][grp], biomass_id=BIOMASS_ID)
        scaled_max[grp] = m.slim_optimize()
        print(f"[{ct}/{grp}] n_scaled: {n_scaled}, scaled_max: {scaled_max[grp]:.4f}")
    shared_growth_rate[ct] = min(scaled_max.values()) * 0.7
    print(f"[{ct}] Shared growth rate: {shared_growth_rate[ct]:.4f}")

print("\nFinal shared growth rates:")
print(shared_growth_rate)

In [ ]:
# Aggregate reaction-level fluxes to pathway (subsystem) level
# Aggregation: SUM of absolute fluxes per pathway (total throughput)
pathway_flux = {}
pathway_summary = {}

EXCLUDE_SUBSYSTEMS = {
    "Miscellaneous", "Pool reactions", "Artificial reactions",
    "Transport reactions", "Exchange/demand reactions",
    "Isolated", "Other",
}

MIN_REACTIONS_PER_PATHWAY = 3

for ct in CELL_TYPES:
    if ct not in flux_matrix:
        continue
    fm = flux_matrix[ct]
    su = summary[ct]

    if "subsystem" not in su.columns:
        print(f"[{ct}] No 'subsystem' column in summary -- skipping pathway aggregation.")
        continue

    reaction_to_subsystem = su["subsystem"].fillna("Unknown")

    # Group |fluxes| by subsystem, SUM per pathway per sample (total throughput)
    fm_abs = fm.abs()
    fm_abs["subsystem"] = reaction_to_subsystem
    pathway_flux[ct] = (
        fm_abs.groupby("subsystem")
              .sum(numeric_only=True)
    )

    counts = reaction_to_subsystem.value_counts()
    keep = (
        ~pathway_flux[ct].index.isin(EXCLUDE_SUBSYSTEMS)
        & pathway_flux[ct].index.map(lambda s: counts.get(s, 0) >= MIN_REACTIONS_PER_PATHWAY)
    )
    pathway_flux[ct] = pathway_flux[ct][keep]

    print(f"[{ct}] Aggregated {fm.shape[0]} reactions into {pathway_flux[ct].shape[0]} pathways "
          f"(metric: sum of |flux| per pathway)")

    if "High" in pathway_flux[ct].columns and "Low" in pathway_flux[ct].columns:
        pathway_summary[ct] = pd.DataFrame({
            "high_total_flux": pathway_flux[ct]["High"],
            "low_total_flux":  pathway_flux[ct]["Low"],
            "group_diff":      pathway_flux[ct]["High"] - pathway_flux[ct]["Low"],
            "n_reactions":     [counts.get(s, 0) for s in pathway_flux[ct].index],
        })
    else:
        high_cols = [c for c in pathway_flux[ct].columns
                     if (isinstance(c, tuple) and c[1] == "High") or "High" in str(c)]
        low_cols  = [c for c in pathway_flux[ct].columns
                     if (isinstance(c, tuple) and c[1] == "Low")  or "Low"  in str(c)]
        pathway_summary[ct] = pd.DataFrame({
            "high_total_flux": pathway_flux[ct][high_cols].mean(axis=1) if high_cols else np.nan,
            "low_total_flux":  pathway_flux[ct][low_cols].mean(axis=1)  if low_cols  else np.nan,
            "n_reactions":     [counts.get(s, 0) for s in pathway_flux[ct].index],
        })
        pathway_summary[ct]["group_diff"] = (
            pathway_summary[ct]["high_total_flux"] - pathway_summary[ct]["low_total_flux"]
        )

    top_pw = pathway_summary[ct].reindex(
        pathway_summary[ct]["group_diff"].abs().sort_values(ascending=False).index
    ).head(15)
    print(f"\n[{ct}] Top pathways by |High - Low| in total absolute flux:")
    print(top_pw.round(3))

    pathway_summary[ct].to_csv(f"{pooled_dir}PathwaySummary_{ct.replace(' ', '')}.csv")

In [ ]:
# heatmap of top pathways by |High - Low| in total absolute flux (pooled)

plot_dir = f"{pooled_dir}Plots/"
os.makedirs(plot_dir, exist_ok=True)

TOP_N = 15  # number of pathways to show

for ct in CELL_TYPES:
    if ct not in pathway_summary or ct not in pathway_flux:
        continue
    ct_tag = ct.replace(" ", "")
    ps = pathway_summary[ct]
    pf = pathway_flux[ct]

    # Pick top pathways by |group_diff|
    top_pw = ps["group_diff"].abs().sort_values(ascending=False).head(TOP_N).index.tolist()
    top_pw = [p for p in top_pw if p in pf.index]

    plot_data = pf.loc[top_pw, ["High", "Low"]]

    # Truncate long pathway names and append reaction count for context
    def wrap(name, width=40):
        return name if len(name) <= width else name[:width-1] + "..."
    row_labels = [
        f"{wrap(p)}  (n={int(ps.loc[p, 'n_reactions'])})"
        for p in plot_data.index
    ]

    # Signed log scale to compress the wide dynamic range
    plot_log = np.sign(plot_data.values) * np.log1p(np.abs(plot_data.values))

    fig, ax = plt.subplots(figsize=(5.5, 0.45 * len(plot_data) + 2))
    sns.heatmap(
        plot_log,
        xticklabels=["High", "Low"],
        yticklabels=row_labels,
        cmap="Reds",
        cbar_kws={"label": "Pathway flux (signed log1p of sum |flux|)"},
        annot=plot_data.round(1).values,  # show raw numbers in cells
        fmt=".1f",
        annot_kws={"size": 8},
        ax=ax,
    )
    ax.set_title(
        f"[{ct}] Pooled pathway-level flux\n"
        f"top {len(plot_data)} pathways by |High - Low| total absolute flux"
    )
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{plot_dir}PathwayHeatmap_{ct_tag}.png", dpi=300)
    plt.show()
    plt.close("all")
    print(f"[{ct}] Saved pathway heatmap to {plot_dir}PathwayHeatmap_{ct_tag}.png")

In [ ]:
# Heatmap of pathway-level High and Low totals, two columns side by side
# Rows ordered by |High - Low| so most differentially active pathways
# remain at the top, but each cell shows the full flux total (not
# just the difference) for that group.


plot_dir = f"{pooled_dir}Plots/"
os.makedirs(plot_dir, exist_ok=True)

TOP_N = 15

for ct in CELL_TYPES:
    if ct not in pathway_summary or ct not in pathway_flux:
        continue
    ct_tag = ct.replace(" ", "")
    ps = pathway_summary[ct]
    pf = pathway_flux[ct]

    # Pick top pathways by absolute group difference
    top_pw_idx = ps["group_diff"].abs().sort_values(ascending=False).head(TOP_N).index.tolist()
    top_pw_idx = [p for p in top_pw_idx if p in pf.index]

    plot_data = pf.loc[top_pw_idx, ["High", "Low"]]

    def wrap(name, width=42):
        return name if len(name) <= width else name[:width-1] + "..."
    row_labels = [
        f"{wrap(p)}  (n={int(ps.loc[p, 'n_reactions'])})"
        for p in plot_data.index
    ]

    # Signed log scale to make the wide dynamic range readable

    plot_log = np.sign(plot_data.values) * np.log1p(np.abs(plot_data.values))

    fig, ax = plt.subplots(figsize=(6.5, 0.45 * len(plot_data) + 2))
    sns.heatmap(
        plot_log,
        xticklabels=["High", "Low"],
        yticklabels=row_labels,
        cmap="Reds",
        cbar_kws={"label": "Pathway flux\n(signed log1p of sum |flux|)"},
        annot=plot_data.round(1).values,   # show raw totals in cells
        fmt=".1f",
        annot_kws={"size": 9},
        ax=ax,
    )
    ax.set_title(
        f"[{ct}] Pooled pathway-level flux\n"
        f"top {len(plot_data)} pathways by |High - Low|, full High and Low totals shown"
    )
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{plot_dir}PathwayHighLowHeatmap_{ct_tag}.png", dpi=300)
    plt.show()
    plt.close("all")
    print(f"[{ct}] Saved side-by-side pathway heatmap to "
          f"{plot_dir}PathwayHighLowHeatmap_{ct_tag}.png")

In [ ]:
# Heatmap of comparative metabolic polarisation
# Both columns show full High and Low totals; colour is mean-centred
# per pathway so direction (up/down relative to row mean) is visible
# alongside raw magnitudes.


plot_dir = f"{pooled_dir}Plots/"
os.makedirs(plot_dir, exist_ok=True)

TOP_N = 25

for ct in CELL_TYPES:
    if ct not in pathway_summary or ct not in pathway_flux:
        continue
    ct_tag = ct.replace(" ", "")
    ps = pathway_summary[ct]
    pf = pathway_flux[ct]

    # Pick top pathways by |group_diff|, keep all of them in display order
    top_idx = ps["group_diff"].abs().sort_values(ascending=False).head(TOP_N).index.tolist()
    top_idx = [p for p in top_idx if p in pf.index]

    plot_data = pf.loc[top_idx, ["High", "Low"]].copy()
    plot_data.columns = ["High S. aureus", "Low S. aureus"]

    # Mean-centred values for colour mapping (per-row centering)
    row_means = plot_data.mean(axis=1)
    plot_centred = plot_data.sub(row_means, axis=0)

    # Symmetric colour scale around 0
    vmax = float(np.abs(plot_centred.values).max())

    fig, ax = plt.subplots(figsize=(8, 0.35 * len(plot_data) + 2))
    sns.heatmap(
        plot_centred.values,
        xticklabels=plot_data.columns,
        yticklabels=plot_data.index,
        cmap="RdBu_r",
        center=0,
        vmin=-vmax, vmax=vmax,
        cbar_kws={"label": "Mean-Centred Flux Shift"},
        annot=plot_data.round(1).values,   # raw flux values shown in cells
        fmt=".1f",
        annot_kws={"size": 9},
        linewidths=0.4,
        linecolor="white",
        ax=ax,
    )
    ax.set_title(
        "Comparative Metabolic Polarisation\n"
        "Red = upregulated relative to row mean, Blue = downregulated",
        fontsize=12, pad=18,
    )
    ax.set_xlabel("Condition Shift", fontsize=11)
    ax.set_ylabel("Metabolic Pathway", fontsize=11)
    plt.xticks(rotation=0)
    plt.subplots_adjust(top=0.92, left=0.45)
    plt.savefig(f"{plot_dir}PathwayPolarisation_{ct_tag}.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    plt.close("all")
    print(f"[{ct}] Saved polarisation heatmap to "
          f"{plot_dir}PathwayPolarisation_{ct_tag}.png")

In [ ]:
for ct in CELL_TYPES:
    if ct in pathway_summary:
        n_total = len(pathway_summary[ct])
        n_meaningful = (pathway_summary[ct]["group_diff"].abs() > 1).sum()
        print(f"[{ct}] {n_total} pathways total, {n_meaningful} with |group_diff| > 1")

In [ ]:
# Re-aggregate wihtout excluding generic subsystems
# Still keeps the MIN_REACTIONS_PER_PATHWAY filter
# subsystems don't appear; only the EXCLUDE_SUBSYSTEMS list is removed.
pathway_flux_full = {}
pathway_summary_full = {}

MIN_REACTIONS_PER_PATHWAY = 3

for ct in CELL_TYPES:
    if ct not in flux_matrix:
        continue
    fm = flux_matrix[ct]
    su = summary[ct]
    if "subsystem" not in su.columns:
        continue

    reaction_to_subsystem = su["subsystem"].fillna("Unknown")

    fm_abs = fm.abs()
    fm_abs["subsystem"] = reaction_to_subsystem
    pf = fm_abs.groupby("subsystem").sum(numeric_only=True)

    counts = reaction_to_subsystem.value_counts()
    keep = pf.index.map(lambda s: counts.get(s, 0) >= MIN_REACTIONS_PER_PATHWAY)
    pf = pf[keep]
    pathway_flux_full[ct] = pf

    if "High" in pf.columns and "Low" in pf.columns:
        ps = pd.DataFrame({
            "high_total_flux": pf["High"],
            "low_total_flux":  pf["Low"],
            "group_diff":      pf["High"] - pf["Low"],
            "n_reactions":     [counts.get(s, 0) for s in pf.index],
        })
    else:
        high_cols = [c for c in pf.columns if "High" in str(c)]
        low_cols  = [c for c in pf.columns if "Low"  in str(c)]
        ps = pd.DataFrame({
            "high_total_flux": pf[high_cols].mean(axis=1) if high_cols else np.nan,
            "low_total_flux":  pf[low_cols].mean(axis=1)  if low_cols  else np.nan,
            "n_reactions":     [counts.get(s, 0) for s in pf.index],
        })
        ps["group_diff"] = ps["high_total_flux"] - ps["low_total_flux"]

    pathway_summary_full[ct] = ps
    print(f"[{ct}] {len(ps)} pathways aggregated (no exclusions)")


#  Polarisation heatmap on the full (unexcluded) pathway set
TOP_N = 25

for ct in CELL_TYPES:
    if ct not in pathway_summary_full or ct not in pathway_flux_full:
        continue
    ct_tag = ct.replace(" ", "")
    ps = pathway_summary_full[ct]
    pf = pathway_flux_full[ct]

    top_idx = ps["group_diff"].abs().sort_values(ascending=False).head(TOP_N).index.tolist()
    top_idx = [p for p in top_idx if p in pf.index]

    plot_data = pf.loc[top_idx, ["High", "Low"]].copy()
    plot_data.columns = ["High S. aureus", "Low S. aureus"]

    row_means = plot_data.mean(axis=1)
    plot_centred = plot_data.sub(row_means, axis=0)
    vmax = float(np.abs(plot_centred.values).max())

    fig, ax = plt.subplots(figsize=(12, 0.35 * len(plot_data) + 2))
    sns.heatmap(
        plot_centred.values,
        xticklabels=plot_data.columns,
        yticklabels=plot_data.index,
        cmap="RdBu_r",
        center=0,
        vmin=-vmax, vmax=vmax,
        cbar_kws={"label": "Mean-Centred Flux Shift"},
        annot=plot_data.round(1).values,
        fmt=".1f",
        annot_kws={"size": 14},
        linewidths=0.4,
        linecolor="white",
        ax=ax,
    )
    cbar = ax.collections[0].colorbar
    cbar.set_label("Mean-Centred Flux Shift", fontsize=16)
    cbar.ax.tick_params(labelsize=14)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=16)
    ax.set_title(
        f"[{ct}] Comparative Metabolic Polarisation\n",
        fontsize=16, pad=18, fontweight = "bold"
    )
    ax.text(
    0.5, 1.02,
    "Red = upregulated relative to row mean, Blue = downregulated",
    transform=ax.transAxes,
    ha="center", va="bottom",
    fontsize=14, fontweight="normal", color="dimgray",
    )
    ax.set_xlabel("Condition Shift", fontsize=16)
    ax.set_ylabel("Metabolic Pathway", fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontweight="bold", fontsize=16)
    plt.subplots_adjust(top=0.92, left=0.45)
    plt.savefig(f"{plot_dir}PathwayPolarisation_full_{ct_tag}.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    plt.close("all")
    print(f"[{ct}] Saved unfiltered polarisation heatmap to "
          f"{plot_dir}PathwayPolarisation_full_{ct_tag}.png")

In [ ]:
# Per-pathway long-form CSV: one row per (pathway × reaction),
# with flux values, metabolites, and associated genes attached.
# One file per cell type.


pathway_csv_dir = f"{pooled_dir}PathwayDetails/"
os.makedirs(pathway_csv_dir, exist_ok=True)

for ct in CELL_TYPES:
    if ct not in flux_matrix or ct not in summary:
        continue
    fm = flux_matrix[ct]
    su = summary[ct]

    rows = []
    for rxn_id in fm.index:
        # Pull reaction metadata from the live cobra model
        try:
            rxn = model.reactions.get_by_id(rxn_id)
        except KeyError:
            continue

        subsystem = su.loc[rxn_id, "subsystem"] if rxn_id in su.index else ""
        name      = su.loc[rxn_id, "name"]      if rxn_id in su.index else rxn.name

        # Metabolites: stoichiometry on each side as a readable string
        # e.g. "1 glc[c] + 1 atp[c] -> 1 g6p[c] + 1 adp[c]"
        substrates = " + ".join(
            f"{-coef:g} {met.id}" for met, coef in rxn.metabolites.items() if coef < 0
        )
        products = " + ".join(
            f"{coef:g} {met.id}"  for met, coef in rxn.metabolites.items() if coef > 0
        )
        arrow = " <=> " if rxn.reversibility else " -> "
        reaction_string = f"{substrates}{arrow}{products}" if substrates or products else ""

        # Genes from GPR (comma-separated symbols/Ensembl IDs)
        genes = ", ".join(sorted(g.id for g in rxn.genes))

        # Per-group flux values (pooled pipeline: columns are "High", "Low")
        flux_values = {f"flux_{col}": fm.loc[rxn_id, col] for col in fm.columns}

        # Group difference (from summary)
        group_diff = su.loc[rxn_id, "group_diff"] if rxn_id in su.index else None

        rows.append({
            "subsystem":      subsystem,
            "reaction_id":    rxn_id,
            "reaction_name":  name,
            "reaction_string": reaction_string,
            **flux_values,
            "group_diff":     group_diff,
            "genes":          genes,
            "n_metabolites":  len(rxn.metabolites),
            "n_genes":        len(rxn.genes),
            "reversible":     rxn.reversibility,
        })

    df = pd.DataFrame(rows)

    # Sort: by subsystem (alphabetical), then by |group_diff| within each subsystem
    df["abs_diff"] = df["group_diff"].abs()
    df = df.sort_values(
        ["subsystem", "abs_diff"],
        ascending=[True, False],
    ).drop(columns="abs_diff")

    ct_tag = ct.replace(" ", "")
    out_path = f"{pathway_csv_dir}PathwayDetails_{ct_tag}.csv"
    df.to_csv(out_path, index=False)
    print(f"[{ct}] Saved {len(df)} reactions across {df['subsystem'].nunique()} pathways "
          f"to {out_path}")

In [ ]:
#  Pathway-level metabolic maps
# For each top pathway:
#   - nodes: metabolites (circles) + reactions (squares)
#   - edges: substrate -> reaction -> product
#   - reaction colour: red = up in High SA, blue = up in Low SA
#   - reaction size: scaled by mean flux magnitude
#   - gene labels: GPR genes annotated next to each reaction

map_dir = f"{pooled_dir}PathwayMaps/"
os.makedirs(map_dir, exist_ok=True)

N_PATHWAYS = 5       # how many top pathways to draw maps for, per cell type
MAX_REACTIONS_PER_MAP = 30  # limit per pathway to keep maps readable

def metabolite_label(met):
    """Strip compartment tag for display, e.g. 'glc[c]' -> 'glc'."""
    name = met.name if met.name else met.id
    return name.replace("[c]", "").replace("[m]", "").replace("[e]", "") \
               .replace("[r]", "").replace("[g]", "").replace("[l]", "") \
               .replace("[n]", "").replace("[x]", "")

for ct in CELL_TYPES:
    if ct not in pathway_summary or ct not in summary or ct not in flux_matrix:
        continue
    ct_tag = ct.replace(" ", "")

    ps = pathway_summary[ct]
    su = summary[ct]
    fm = flux_matrix[ct]

    # Pick top N pathways by |group_diff|, but skip the generic ones
    EXCLUDE = {"Transport reactions", "Exchange/demand reactions",
               "Miscellaneous", "Pool reactions", "Artificial reactions"}
    candidates = ps[~ps.index.isin(EXCLUDE)]
    # keep only pathways that show differential flux signal, removing ones with
    # equal flux

    top_pathways = candidates["group_diff"].abs().sort_values(ascending=False).head(N_PATHWAYS).index.tolist()


    for pw in top_pathways:
        # Get all reactions in this pathway, sorted by |group_diff|
        rxn_ids = su[su["subsystem"] == pw].index.tolist()
        rxn_ids_ranked = sorted(
            rxn_ids,
            key=lambda r: abs(su.loc[r, "group_diff"]) if pd.notna(su.loc[r, "group_diff"]) else 0,
            reverse=True,
        )[:MAX_REACTIONS_PER_MAP]

        if len(rxn_ids_ranked) < 2:
            continue

        # Build a bipartite graph
        G = nx.DiGraph()
        rxn_data = {}    # rxn_id -> {flux_diff, mean_flux, genes}

        for rxn_id in rxn_ids_ranked:
            try:
                rxn = model.reactions.get_by_id(rxn_id)
            except KeyError:
                continue

            diff = su.loc[rxn_id, "group_diff"]
            mean_flux = (abs(fm.loc[rxn_id, "High"]) + abs(fm.loc[rxn_id, "Low"])) / 2
            genes = ", ".join(sorted(g.id for g in rxn.genes))

            rxn_data[rxn_id] = {"diff": diff, "mean_flux": mean_flux, "genes": genes}
            G.add_node(rxn_id, node_type="reaction")

            for met, coef in rxn.metabolites.items():
                G.add_node(met.id, node_type="metabolite", display=metabolite_label(met))
                if coef < 0:
                    G.add_edge(met.id, rxn_id)
                else:
                    G.add_edge(rxn_id, met.id)

        if len(G.nodes) == 0:
            continue

        # Layout
        pos = nx.spring_layout(G, k=1.2, iterations=80, seed=42)

        # Set up node attributes for drawing
        rxn_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "reaction"]
        met_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "metabolite"]

        # Symmetric colour scale around 0 for reactions
        diffs = [rxn_data[r]["diff"] for r in rxn_nodes]
        vmax = max(abs(min(diffs)), abs(max(diffs)), 1e-6) # 1e-6 floor avoids
                                                              # divide-by-zero if all
                                                              # diffs happen to be 0

        # Reaction node sizes from mean flux magnitude
        mean_fluxes = [rxn_data[r]["mean_flux"] for r in rxn_nodes]
        max_flux = max(mean_fluxes) if mean_fluxes else 1
        # Scale node area between 200 (min, always visible) and 1000 (200+800,
        # largest flux reaction) so size differences are visible without any
        # node disappearing entirely

        rxn_sizes = [200 + 800 * (f / max_flux) for f in mean_fluxes]

        fig, ax = plt.subplots(figsize=(13, 11))

        # Metabolites
        nx.draw_networkx_nodes(
            G, pos, nodelist=met_nodes,
            node_color="lightgrey", node_shape="o", node_size=120,
            edgecolors="dimgrey", linewidths=0.5, ax=ax,
        )
        # Reactions
        nx.draw_networkx_nodes(
            G, pos, nodelist=rxn_nodes,
            node_color=diffs,
            cmap="RdBu_r", vmin=-vmax, vmax=vmax,
            node_shape="s", node_size=rxn_sizes,
            edgecolors="black", linewidths=0.8, ax=ax,
        )
        # Edges
        nx.draw_networkx_edges(
            G, pos, alpha=0.35, arrows=True, arrowsize=8,
            edge_color="grey", width=0.6, ax=ax,
        )

        # Metabolite labels (small, light grey)
        met_labels = {n: G.nodes[n].get("display", n) for n in met_nodes}
        nx.draw_networkx_labels(
            G, pos, labels=met_labels, font_size=7,
            font_color="dimgrey", ax=ax,
        )

       # Gene labels only if the combined symbol string is short enough to fit
        # on the node (<30 chars) - otherwise fall back to the reaction ID to
        # avoid overlapping label clutter

        rxn_labels = {
            r: (rxn_data[r]["genes"] if rxn_data[r]["genes"] and len(rxn_data[r]["genes"]) < 30 else r)
            for r in rxn_nodes
        }
        nx.draw_networkx_labels(
            G, pos, labels=rxn_labels, font_size=8, font_weight="bold",
            font_color="black", ax=ax,
        )

        sm = plt.cm.ScalarMappable(cmap="RdBu_r",
                                   norm=plt.Normalize(vmin=-vmax, vmax=vmax))
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
        cbar.set_label("Reaction flux difference\n(High - Low)", fontsize=10)

        # Legend for node shape
        legend_elements = [
            mpatches.Patch(facecolor="lightgrey", edgecolor="dimgrey", label="Metabolite"),
            mpatches.Patch(facecolor="white", edgecolor="black", label="Reaction (size = mean |flux|)"),
        ]
        ax.legend(handles=legend_elements, loc="lower left", fontsize=9, frameon=True)

        ax.set_title(
            f"[{ct}]  {pw}\n"
            f"Metabolic map of top {len(rxn_nodes)} reactions in this pathway",
            fontsize=12, fontweight="bold",
        )
        ax.axis("off")

        out_path = f"{map_dir}Map_{ct_tag}_{pw[:40].replace('/', '_').replace(' ', '_')}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close("all")
        print(f"[{ct}] Saved map for '{pw}' to {out_path}")

In [ ]:
# Per-pathway gene-driver tables
# For each top pathway, extract the genes driving each reaction's flux
# difference


gene_csv_dir = f"{pooled_dir}PathwayGeneDrivers/"
os.makedirs(gene_csv_dir, exist_ok=True)

ens_to_symbol = {}
if "ens_to_symbol" in dir():
    pass

else:
    # Try to invert symbol_to_ens if it exists
    try:
        ens_to_symbol = {v: k for k, v in symbol_to_ens.items()}
    except (NameError, AttributeError):
        print("Warning: no Ensembl-to-symbol mapping found; genes will be shown as Ensembl IDs.")

def symbolify(gene_ids_str):
    """Convert a comma-separated string of Ensembl IDs to gene symbols where possible."""
    if not gene_ids_str:
        return ""
    out = []
    for g in [x.strip() for x in gene_ids_str.split(",")]:
        out.append(ens_to_symbol.get(g, g))
    return ", ".join(out)


for ct in CELL_TYPES:
    if ct not in pathway_summary or ct not in summary:
        continue
    ct_tag = ct.replace(" ", "")

    ps = pathway_summary[ct]
    su = summary[ct]


    candidates = ps[~ps.index.isin(EXCLUDE)]
    # keep only pathways that show differential flux signal, removing ones with equal flux to prevent them cluttering the file
    top_pathways = candidates[candidates["group_diff"].abs() > 1.0].index.tolist()

    all_rows = []

    for pw in top_pathways:
        rxn_ids = su[su["subsystem"] == pw].index.tolist()

        for rxn_id in rxn_ids:
            try:
                rxn = model.reactions.get_by_id(rxn_id)
            except KeyError:
                continue
            if not rxn.genes:
                continue

            group_diff = su.loc[rxn_id, "group_diff"]
            high_flux  = flux_matrix[ct].loc[rxn_id, "High"] if rxn_id in flux_matrix[ct].index else None
            low_flux   = flux_matrix[ct].loc[rxn_id, "Low"]  if rxn_id in flux_matrix[ct].index else None
            rxn_name   = su.loc[rxn_id, "name"] if rxn_id in su.index else rxn.name

            # Expand to one row per (gene × reaction)
            for gene in rxn.genes:
                gene_id = gene.id
                gene_symbol = ens_to_symbol.get(gene_id, gene_id)

                all_rows.append({
                    "pathway":        pw,
                    "gene_symbol":    gene_symbol,
                    "gene_ensembl":   gene_id,
                    "reaction_id":    rxn_id,
                    "reaction_name":  rxn_name,
                    "high_flux":      high_flux,
                    "low_flux":       low_flux,
                    "flux_diff":      group_diff,
                    "direction":      ("up_in_High" if group_diff > 0
                                       else ("down_in_High" if group_diff < 0 else "no_change")),
                    "abs_flux_diff":  abs(group_diff) if pd.notna(group_diff) else 0,
                })

    df = pd.DataFrame(all_rows)

    # Aggregate per gene: a gene may control multiple reactions in the same pathway
    gene_summary = (df.groupby(["pathway", "gene_symbol", "gene_ensembl"])
                      .agg(
                          n_reactions = ("reaction_id", "nunique"),
                          mean_flux_diff = ("flux_diff", "mean"),
                          max_abs_flux_diff = ("abs_flux_diff", "max"),
                          reactions = ("reaction_id", lambda x: ", ".join(sorted(set(x)))),
                          reaction_names = ("reaction_name", lambda x: " | ".join(sorted(set(str(v) for v in x if pd.notna(v))))),
                      )
                      .reset_index()
                      .sort_values(["pathway", "max_abs_flux_diff"], ascending=[True, False]))

    gene_summary["direction"] = gene_summary["mean_flux_diff"].apply(
        lambda x: "up_in_High" if x > 0 else ("down_in_High" if x < 0 else "no_change"))

    # Save the long-form (per gene-reaction pair) and per-gene-summary versions
    df.to_csv(f"{gene_csv_dir}GeneDrivers_long_{ct_tag}.csv", index=False)
    gene_summary.to_csv(f"{gene_csv_dir}GeneDrivers_perGene_{ct_tag}.csv", index=False)
    print(f"[{ct}] Saved {len(df)} (gene × reaction) rows and "
          f"{len(gene_summary)} unique genes across {gene_summary['pathway'].nunique()} pathways")

In [ ]:
#  Gene-driver barplot per top pathway
gene_plot_dir = f"{pooled_dir}PathwayGeneDrivers/Plots/"
os.makedirs(gene_plot_dir, exist_ok=True)

TOP_GENES_PER_PATHWAY = 15 # genes per subplot - chosen to stay readable at the
                             # font size used below without the bars overlapping
ens_to_symbol = {v: k for k, v in symbol_to_ens.items()}
print(f"Built ens_to_symbol mapping for {len(ens_to_symbol)} genes")

unmapped = [g.id for g in model.genes if g.id not in ens_to_symbol]
print(f"{len(unmapped)} Human-GEM genes have no symbol in the current mapping")

# Second-pass query for the unmapped ones, going from Ensembl -> symbol
# The first mapping (symbol_to_ens, built earlier) only covers genes that were
# successfully queried by symbol; some Human-GEM genes use Ensembl IDs that
# weren't in that original query, so we look those up directly here instead.

if unmapped:
    mapping2 = mg.querymany(unmapped, scopes='ensembl.gene', fields='symbol', species='human')
    for entry in mapping2:
        if entry.get('notfound'):
            continue
        if 'symbol' in entry:
            ens_to_symbol[entry['query']] = entry['symbol']
    print(f"After second pass: {len(ens_to_symbol)} mapped")

for ct in CELL_TYPES:
    if ct not in pathway_summary:
        continue
    ct_tag = ct.replace(" ", "")
    ps = pathway_summary[ct]

    gs = pd.read_csv(f"{gene_csv_dir}GeneDrivers_perGene_{ct_tag}.csv")

    candidates = ps[~ps.index.isin(EXCLUDE)]
    top_pathways = (candidates["group_diff"].abs()
                    .sort_values(ascending=False).head(6).index.tolist())  # 6 to fit on a 2x3 grid

    fig, axes = plt.subplots(2, 3, figsize=(18, 16))
    axes = axes.flatten()

    for i, pw in enumerate(top_pathways):
        ax = axes[i]
        pw_genes = gs[gs["pathway"] == pw].copy()
        pw_genes["gene_symbol"] = pw_genes["gene_ensembl"].map(ens_to_symbol).fillna(pw_genes["gene_symbol"])
        pw_genes = pw_genes.reindex(
            pw_genes["max_abs_flux_diff"].sort_values(ascending=False).index
        ).head(TOP_GENES_PER_PATHWAY)

        if pw_genes.empty:
            ax.axis("off")
            continue

        colours = ["#c0392b" if d == "up_in_High" else "#2980b9" for d in pw_genes["direction"]]
        ax.barh(range(len(pw_genes)), pw_genes["mean_flux_diff"], color=colours)
        ax.set_yticks(range(len(pw_genes)))
        ax.set_yticklabels(pw_genes["gene_symbol"], fontsize=16)
        ax.invert_yaxis()
        ax.axvline(0, color="black", linewidth=0.5)
        ax.set_xlabel("Mean flux difference (High - Low)", fontsize=18)
        ax.tick_params(axis='x', labelsize=16)

        # Truncate long pathway titles
        wrapped_title = textwrap.fill(pw, width=30, break_long_words=False)
        ax.set_title(wrapped_title, fontsize=16, fontweight="bold", pad=15)

    # Hide any unused subplots
    for j in range(len(top_pathways), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        f"[{ct}] Top gene drivers per pathway\n"
        f"Red = upregulated in High SA, Blue = downregulated",
        fontsize=20, fontweight="bold", y=1.0,
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.subplots_adjust(hspace=0.3)
    plt.savefig(f"{gene_plot_dir}GeneDrivers_Grid_{ct_tag}.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    plt.close("all")
    print(f"[{ct}] Saved gene-driver grid to {gene_plot_dir}GeneDrivers_Grid_{ct_tag}.png")